In [65]:
import torch
from torch import nn

from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np

from typing import Tuple, List, Optional
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime



import lightning as pl


from Models.models import Siren, Finer
from Utils.utils import get_full_img, norm, get_device, dice_stack_helper, get_model, ClearCache
from Data.load_data_3d import load_data, get_gt_seg
from Utils.defaults import default_config
from Utils.plotting_utils2 import plot_seg_results_paper, plot_final_results_paper, plot_hf_results_paper
from Utils.plotting_utils import loss_plot, plot_image_metrics, plot_4_images
from LFSynth.ContrastModulation import ContrastModulation
from test3D import visualize_volume_slices
import copy

In [88]:
config = copy.deepcopy(default_config)
config["in_features"] = 3
hf_ground_truth, lf_gt, prior_seg_dice, lf_gt_seg_dice, M = load_data(1, config) #uncomment
gt_image = torch.tensor(norm(hf_ground_truth)).unsqueeze(-1)
gt_image = gt_image.to(torch.float32)

lf_gt = torch.tensor(norm(lf_gt)).unsqueeze(-1)
lf_gt = lf_gt.to(torch.float32)
print('gt_image loaded')

Device =  mps
torch.Size([87, 96, 192])
BG Noise in different regions : 4.7445857300717 4.415978248235479 4.956849146645869 4.8247938816077935
known_m =  [0.75 0.9  0.9 ]
[[ 28.84998854   0.          -7.89492106]
 [ 28.84998854 -22.35619149   0.        ]
 [  0.          22.35619149  -7.89492106]] [14.53206245  1.51691906 13.01514339]
             m_init  epsilon  \
0   [0.1, 0.1, 0.1]     0.00   
1   [0.1, 0.1, 0.1]     0.01   
2   [0.1, 0.1, 0.1]     0.05   
3   [0.1, 0.1, 0.1]     0.10   
4   [0.1, 0.1, 0.1]     0.20   
..              ...      ...   
59  [0.5, 0.5, 0.5]     0.10   
60  [0.5, 0.5, 0.5]     0.20   
61  [0.5, 0.5, 0.5]     0.30   
62  [0.5, 0.5, 0.5]     0.40   
63  [0.5, 0.5, 0.5]     0.50   

                                                 loss  \
0   [144.631340676391, 144.63133495698293, 144.631...   
1   [144.63164067639102, 144.63163495701272, 144.6...   
2   [144.632840676391, 144.63283495713193, 144.632...   
3   [144.634340676391, 144.63433495728094, 144.634.

In [181]:
type(lf_gt), lf_gt.shape, gt_image.shape, type(gt_image), lf_gt.dtype, gt_image.dtype

(torch.Tensor,
 torch.Size([87, 96, 192, 1]),
 torch.Size([174, 192, 192, 1]),
 torch.Tensor,
 torch.float32,
 torch.float32)

In [182]:
lf_gt.max(), lf_gt.min(), gt_image.max(), gt_image.min()

(tensor(1.), tensor(0.), tensor(1.), tensor(0.))

randompoints for unsupervised setup

In [332]:
class RandomPointsDataset(Dataset):
    def __init__(self, image: torch.Tensor, lf_image:torch.Tensor, points_num: int = POINTS_PER_SAMPLE):
        super().__init__()
        self.device = 'cpu' #get_device()
        self.points_num = points_num
        assert image.dtype == torch.float32
        assert lf_image.dtype == torch.float32
        self.image = image.to(self.device)  # (H, W, ..., C)
        self.lf_image = lf_image.to(self.device)  # (H, W, ..., C)
        # self.dim_sizes = self.image.shape[:-1]  # Size of each spatial dimension
        self.dim_sizes = self.lf_image.shape[:-1]  # Size of each spatial dimension
        

        # To help us define the input/output sizes of our network later
        # we store the size of our input coordinates and output values
        self.coord_size = len(self.image.shape[:-1])  # Number of spatial dimensions
        self.value_size = self.lf_image.shape[-1]  # Channel size
        # self.value_size = self.lf_image.shape[-1]  # Channel size

    def __len__(self):
        return 1

    # def __getitem__(self, idx: int):
    #     # Create random sample of pixel indices
    #     point_indices = [torch.randint(0, i, (self.points_num,), device=self.device) for i in self.dim_sizes]
    #     # point_indices = [i.to(torch.int32) for i in point_indices]
    #     # point_indices = [i.to('cpu') for i in point_indices]
    #     # print(point_indices[0].dtype, point_indices[0].device)
        
    #     point_indices_lf = [(F.interpolate(i.unsqueeze(0).unsqueeze(0).to(torch.float32), scale_factor=0.25)).squeeze(0).squeeze(0) for i in point_indices]
    #     point_indices = [i.to(torch.int64) for i in point_indices]
    #     point_indices_lf = [i.to(torch.int64) for i in point_indices_lf]

    #     print(point_indices[0], point_indices_lf[0])
    #     # Retrieve image values from selected indices
    #     point_values = self.image[tuple(point_indices)]
    #     print(self.image.shape, point_values.shape, self.lf_image.shape)
    #     point_values_lf = self.lf_image[tuple(point_indices_lf)]
    #     # print(point_values.shape, point_values_lf.shape)
        
    #     # Convert point indices into normalized [-1.0, 1.0] coordinates
    #     point_coords = torch.stack(point_indices, dim=-1)
    #     spatial_dims = torch.tensor(self.dim_sizes, device=self.device)
    #     point_coords_norm = point_coords / (spatial_dims / 2) - 1

    #     # The subject index is also returned in case the user wants to use subject-wise learned latents
    #     return point_coords_norm, point_values#, point_values_lf

    def __getitem__(self, idx: int):
        # # Create random sample of pixel indices
        
        point_indices = [torch.randint(0, i, (self.points_num,), device=self.device) for i in self.dim_sizes]
        print(point_indices[0].shape)
        # Retrieve image values from selected indices
        point_values = self.lf_image[tuple(point_indices)]
        print(point_values.shape)
        point_values = F.interpolate(point_values.unsqueeze(0).unsqueeze(0).squeeze(-1), scale_factor=0.25).squeeze(0).squeeze(0)
        print(point_values.shape)
        # Convert point indices into normalized [-1.0, 1.0] coordinates
        point_coords = torch.stack(point_indices, dim=-1)
        spatial_dims = torch.tensor(self.dim_sizes, device=self.device)
        point_coords_norm = point_coords / (spatial_dims / 2) - 1

        # The subject index is also returned in case the user wants to use subject-wise learned latents
        return point_coords_norm, point_values
        


torch.Size([36864])
torch.Size([36864, 1])
torch.Size([9216])


randompoints for simultaneous unsupervised and segmentation setup

In [339]:
lf_gt_seg_dice.shape

torch.Size([1, 4, 87, 96, 192])

In [358]:
class RandomPointsDataset(Dataset):
    def __init__(self, image: torch.Tensor, lf_image:torch.Tensor, lf_gt_seg_dice:torch.Tensor, points_num: int = POINTS_PER_SAMPLE):
        super().__init__()
        self.device = 'cpu' #get_device()
        self.points_num = points_num
        assert image.dtype == torch.float32
        assert lf_image.dtype == torch.float32
        self.image = image.to(self.device)  # (H, W, ..., C)
        self.lf_image = lf_image.to(self.device)  # (H, W, ..., C)
        self.lf_gt_seg_dice = lf_gt_seg_dice.permute(1,2,3,4,0).to(self.device) #(tissues, H,W,D, C)
        # self.dim_sizes = self.image.shape[:-1]  # Size of each spatial dimension
        self.dim_sizes = self.lf_image.shape[:-1]  # Size of each spatial dimension
        

        # To help us define the input/output sizes of our network later
        # we store the size of our input coordinates and output values
        self.coord_size = len(self.image.shape[:-1])  # Number of spatial dimensions
        self.value_size = self.lf_image.shape[-1]  # Channel size
        # self.value_size = self.lf_image.shape[-1]  # Channel size

    def __len__(self):
        return 1


    def __getitem__(self, idx: int):
        # # Create random sample of pixel indices
        
        point_indices = [torch.randint(0, i, (self.points_num,), device=self.device) for i in self.dim_sizes]
        # print(point_indices[0].shape)
        # Retrieve image values from selected indices
        point_values = self.lf_image[tuple(point_indices)]

        point_values_seg = [self.lf_gt_seg_dice[i][tuple(point_indices)] for i in range(self.lf_gt_seg_dice.shape[0])]
        point_values_seg = torch.stack(point_values_seg,axis = 0)
        # print(point_values.shape, point_values_seg.shape)
        point_values = F.interpolate(point_values.unsqueeze(0).unsqueeze(0).squeeze(-1), scale_factor=0.25).squeeze(0).squeeze(0) #downsampling lf_gt
        point_values_seg = [F.interpolate(point_values_seg[i].unsqueeze(0).unsqueeze(0).squeeze(-1), scale_factor=0.25).squeeze(0).squeeze(0) for i in range(self.lf_gt_seg_dice.shape[0])] #downsampling lf_gt_seg
        point_values_seg = torch.stack(point_values_seg,axis = 0)
        # print(point_values.shape, point_values_seg.shape)

        # Convert point indices into normalized [-1.0, 1.0] coordinates
        point_coords = torch.stack(point_indices, dim=-1)
        spatial_dims = torch.tensor(self.dim_sizes, device=self.device)
        point_coords_norm = point_coords / (spatial_dims / 2) - 1

        # The subject index is also returned in case the user wants to use subject-wise learned latents
        return point_coords_norm, point_values
        
POINTS_PER_SAMPLE = 96*96*4
dataset = RandomPointsDataset(gt_image, lf_gt, lf_gt_seg_dice, points_num=POINTS_PER_SAMPLE)
dataloader = DataLoader(dataset, batch_size=1, num_workers=0, pin_memory=False) # We set a batch_size of 1 since our dataloader is already returning a batch of points.
temp = next(iter(dataloader))


In [367]:
dataset.coord_size, dataset.value_size

(3, 1)

In [351]:
lf_gt_seg_dice.shape[1]
for i in range()
    

4

In [366]:
class MyModule(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linears = nn.ModuleList([nn.Linear(10, 10) for i in range(10)])

    def forward(self, x):
        # ModuleList can act as an iterable, or be indexed using ints
        for i, l in enumerate(self.linears[:-1]):
            print(i, l)
            x = self.linears[i // 2](x) + l(x)
        t1 = x[0]
        t2 = x[1:]

        return t1, t2

temp1 = MyModule()
x = torch.randn(10, 10 )
op1, op2 = temp1.forward(x)
op1.shape, op2.shape

0 Linear(in_features=10, out_features=10, bias=True)
1 Linear(in_features=10, out_features=10, bias=True)
2 Linear(in_features=10, out_features=10, bias=True)
3 Linear(in_features=10, out_features=10, bias=True)
4 Linear(in_features=10, out_features=10, bias=True)
5 Linear(in_features=10, out_features=10, bias=True)
6 Linear(in_features=10, out_features=10, bias=True)
7 Linear(in_features=10, out_features=10, bias=True)
8 Linear(in_features=10, out_features=10, bias=True)


(torch.Size([10]), torch.Size([9, 10]))

In [395]:
class RandomPointsDataset(Dataset):
    def __init__(self, image: torch.Tensor, lf_image:torch.Tensor, lf_gt_seg_dice:torch.Tensor, points_num: int = POINTS_PER_SAMPLE):
        super().__init__()
        self.device = 'cpu' #get_device()
        self.points_num = points_num
        assert image.dtype == torch.float32
        assert lf_image.dtype == torch.float32
        self.image = image.to(self.device)  # (H, W, ..., C)
        self.lf_image = lf_image.to(self.device)  # (H, W, ..., C)
        self.lf_gt_seg_dice = lf_gt_seg_dice.permute(1,2,3,4,0).to(self.device) #(tissues, H,W,D, C)
        # self.dim_sizes = self.image.shape[:-1]  # Size of each spatial dimension
        self.dim_sizes = self.lf_image.shape[:-1]  # Size of each spatial dimension
        

        # To help us define the input/output sizes of our network later
        # we store the size of our input coordinates and output values
        self.coord_size = len(self.image.shape[:-1])  # Number of spatial dimensions
        self.value_size = self.lf_image.shape[-1]  # Channel size
        # self.value_size = self.lf_image.shape[-1]  # Channel size

    def __len__(self):
        return 1


    def __getitem__(self, idx: int):
        # # Create random sample of pixel indices
        
        point_indices = [torch.randint(0, i, (self.points_num,), device=self.device) for i in self.dim_sizes]
        # print(point_indices[0].shape)
        # Retrieve image values from selected indices
        point_values = self.lf_image[tuple(point_indices)]

        point_values_seg = [self.lf_gt_seg_dice[i][tuple(point_indices)] for i in range(self.lf_gt_seg_dice.shape[0])]
        point_values_seg = torch.stack(point_values_seg,axis = 0)
        # print(point_values.shape, point_values_seg.shape)
        point_values = F.interpolate(point_values.unsqueeze(0).unsqueeze(0).squeeze(-1), scale_factor=0.25).squeeze(0).squeeze(0) #downsampling lf_gt
        point_values_seg = [F.interpolate(point_values_seg[i].unsqueeze(0).unsqueeze(0).squeeze(-1), scale_factor=0.25).squeeze(0).squeeze(0) for i in range(self.lf_gt_seg_dice.shape[0])] #downsampling lf_gt_seg
        point_values_seg = torch.stack(point_values_seg,axis = 0)
        # print(point_values.shape, point_values_seg.shape)

        # Convert point indices into normalized [-1.0, 1.0] coordinates
        point_coords = torch.stack(point_indices, dim=-1)
        spatial_dims = torch.tensor(self.dim_sizes, device=self.device)
        point_coords_norm = point_coords / (spatial_dims / 2) - 1

        # The subject index is also returned in case the user wants to use subject-wise learned latents
        return point_coords_norm, point_values, point_values_seg

dataset = RandomPointsDataset(gt_image, lf_gt, lf_gt_seg_dice, points_num=POINTS_PER_SAMPLE)
dataloader = DataLoader(dataset, batch_size=1, num_workers=0, pin_memory=False) # We set a batch_size of 1 since our dataloader is already returning a batch of points.
    


In [398]:
POINTS_PER_SAMPLE = 96*96*4
lf_points_per_sample = 48*48*4
class ReLULayer(nn.Module):
    def __init__(self,
                 in_size: int,
                 out_size: int,
                 **kwargs):
        super().__init__()
        self.linear = nn.Linear(in_size, out_size, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        return x


class MLP(nn.Module):
    def __init__(self,
                 in_size: int,
                 out_size: int,
                 hidden_size: int = 128,
                 num_layers: int = 3,
                 layer_class: nn.Module = ReLULayer,
                 **kwargs):
        super().__init__()

        a = [layer_class(in_size, hidden_size, **kwargs)]
        for i in range(num_layers - 1):
            a.append(layer_class(hidden_size, hidden_size, **kwargs))
        
        # a.append(nn.Linear(hidden_size, 1)) #actual image
        a.append(nn.Linear(hidden_size, out_size)) #segmentations+image
        self.layers = nn.ModuleList(a)        
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=-1)
        print("MLP init. complete")

    def forward(self, x: torch.Tensor):
        for i, layer in enumerate(self.layers):
            x = layer(x)
        print("FW pass: hidden layers ")
        print(x.shape)
        output_image_pre = x[:,0] #output image before applying activation function
        output_seg_pre = x[:,1:] #output seg before applying activation function
        output_image = self.relu(output_image_pre)
        output_seg = self.softmax(output_seg_pre)
        print(output_image.shape, output_seg.shape)
        return output_image, output_seg


siren_inr = MLP(in_size = 3,
                out_size = 5, 
                hidden_size=32, num_layers=3,
                layer_class=ReLULayer, 
                siren_factor=30.0,
                )

temp_input, _,  _ = next(iter(dataloader))
temp_input = temp_input.view(-1, temp_input.shape[-1])
print(temp_input.shape)
temp_op = siren_inr(temp_input)

MLP init. complete
torch.Size([36864, 3])
FW pass: hidden layers 
torch.Size([36864, 5])
torch.Size([36864]) torch.Size([36864, 4])


In [394]:
temp_input.shape

torch.Size([36864, 3])

In [387]:
temp_op[0].shape, temp_op[1].shape

(torch.Size([1, 36864]), torch.Size([1, 36864, 4]))

In [400]:
from monai.losses.dice import *  # NOQA
import torch
from monai.losses.dice import DiceLoss
B, C, H, W, D = 7, 5, 3, 2, 5
input = torch.rand(B, C, H, W, D)

target_idx = torch.randint(low=0, high=C - 1, size=(B, H, W)).long()
target = one_hot(target_idx[:, None, ...], num_classes=C)
self = DiceLoss(reduction='none')
print(input.shape, target.shape)
loss = self(input, target)

torch.Size([7, 5, 3, 2, 5]) torch.Size([7, 5, 3, 2])


AssertionError: ground truth has different shape (torch.Size([7, 5, 3, 2])) from input (torch.Size([7, 5, 3, 2, 5]))

In [419]:
from kornia.losses import total_variation
a = torch.randn(36864)
total_variation(a.reshape(96,96,4), reduction='mean').mean()
print(total_variation(a.reshape(96,96,4), reduction='mean').mean())
print(total_variation(a.reshape(4,96,96), reduction='mean').mean())

In [424]:
a = [1,4,48,48,48]
a[2:]

[48, 48, 48]